In [ ]:
from dataclasses import dataclass

In [ ]:
@dataclass
class Polynumber:
    coeffs: dict # {(exponent,): value}, }

    def __repr__(self):
        return str(self.coeffs)

    def __post_init__(self):
        self.coeffs = {
            k: v
            for k, v in self.coeffs.items()
            if isinstance(k, tuple)
            and isinstance(k[0], int)
            and isinstance(v, (int, float))
            and v != 0
        }

    def __eq__(self, other):
        # cannot compare Polynumber to another type
        if not isinstance(other, Polynumber):
            return False
        
        if self.coeffs == other.coeffs:
            return True
        return False
    
    def __ne__(self, other):
        return not self.__eq__(other)

    def __add__(self, other):
        # cannot add Polynumber to another type
        if not isinstance(other, Polynumber):
            return NotImplemented
        
        all_coeff_keys = {*self.coeffs.keys(), *other.coeffs.keys()}
        
        # 2D+ case.  not yet implemented
        if any(len(k) > 1 for k in all_coeff_keys):
            return NotImplemented
        
        # 1D case
        if not any(len(k) > 1 for k in all_coeff_keys):
            return Polynumber({c: (self.coeffs.get(c, 0) + other.coeffs.get(c, 0)) for c in all_coeff_keys})

    def __radd__(self, other):
        return self.__add__(other)

    def __sub__(self, other):
        return self + (other * -1)

    def __rsub__(self, other):
        return (self * -1) + other

    def __rmul__(self, other):
        return self.__mul__(other)

    def __mul__(self, other):
        # can multiply Polynumber to an integer
        if isinstance(other, int):
            return Polynumber({e: (v * other) for e, v in self.coeffs.items()})

        # cannot multiply Polynumber to another type
        if not isinstance(other, Polynumber):
            return NotImplemented
        
        all_coeff_keys = {*self.coeffs.keys(), *other.coeffs.keys()}
        
        # 2D+ case.  not yet implemented
        if any(len(k) > 1 for k in all_coeff_keys):
            return NotImplemented
        
        # 1D case
        if not any(len(k) > 1 for k in all_coeff_keys):
            # parts will be {sum of exponent pair: [product of a value pair, product of next value pair]}
            parts = {}
            for e1, v1 in self.coeffs.items():
                for e2, v2 in other.coeffs.items():
                    exponent_pair = e1[0] + e2[0]
                    value_pair = v1 * v2 
                    if not parts.get(exponent_pair):
                        parts[exponent_pair] = [value_pair]
                    else:
                        parts[exponent_pair].append(value_pair)
            return Polynumber({(e,): sum(v) for e, v in parts.items()})
        
    def __truediv__(self, other):
        # print('self:', self)
        # print('other:', other)

        # cant divivde by 0
        if not other.coeffs:
            return NotImplemented

        # can divide Polynumber by an integer      
        if isinstance(other, int):
            new = Polynumber({e: (v / other) for e, v in self.coeffs.items()})
            # no plan yet for when a coeff value / the divisor is non-integer
            for v in new.coeffs.values():
                if not isinstance(v, int):
                    return NotImplemented
            return new

        # cannot divide Polynumber by another type
        if not isinstance(other, Polynumber):
            return NotImplemented

        all_coeff_keys = {*self.coeffs.keys(), *other.coeffs.keys()}

        # 2D+ case.  not yet implemented
        if any(len(k) > 1 for k in all_coeff_keys):
            return NotImplemented

        # 1D case
        if not any(len(k) > 1 for k in all_coeff_keys):
            result = Polynumber({})
            max_self_exponent = max([i[0] for i in self.coeffs.keys()])
            print("max_self_exponent", max_self_exponent)

            # Keep track of the carry-over remainder between steps.
            remainder = self

            for i in range(max_self_exponent + 1):
                if i == 0:
                    dividend = self
                    divisor = other
                else:
                    dividend = remainder
                    divisor = other * Polynumber({(1,): 1})

                # found the escape.  dividend is 0
                if dividend * 0 == dividend:
                    print('found dividend is 0')
                    break

                e = (i, )
                print("e:", e)
                print("dividend:", dividend)
                print("divisor:", divisor)
                v = divisor.coeffs.get(e)
                # print("v:", v)
                print("dividend.coeffs.get(e):", dividend.coeffs.get(e))
                if not dividend.coeffs.get(e) or not v:
                    continue
                max_division = dividend.coeffs[e] // v
                print("max_division:", max_division)

                for mult in range(max_division + 1):
                    print("mult:", mult)
                    # minuend - subtrahend = remainder
                    minuend = dividend
                    subtrahend = mult * divisor
                    remainder = minuend - subtrahend
                    print("minuend:", minuend)
                    print("subtrahend:", subtrahend)
                    print("remainder:", remainder)

                    if not remainder.coeffs.get(e):
                        result.coeffs[e] = mult
                        print("result", result)
                        break
            return result

    def __call__(self, val):
        if isinstance(val, int):
            return sum(v * val**e[0] for e, v in self.coeffs.items())

        return NotImplemented

In [ ]:
p = Polynumber({(0,): 2, (1,): 7, (2,): 2, (3,): -3})
q = Polynumber({(0,): 4})
r = Polynumber({(0,): 2, (1,): 1, (2,): -1})

In [ ]:
from random import randint

def setup():
    dim1 = randint(0, 5)
    dim2 = randint(0, 5)
    a = Polynumber({(d,): randint(0, 20) for d in range(dim1)})
    b = Polynumber({(d,): randint(0, 20) for d in range(dim2)})
    return a, b

def test(a, b):
    if not a.coeffs or not b.coeffs:
        print("pass -- cannot divide by 0")
        return

    print("a", a)
    print("b", b)
    c = a * b
    print("c", c)
    print("doing c / a")
    d = c / a
    if d != b:
        print(f"fail at {c} / {a} != {b}.  == {d}")
        return False
    else:
        print("pass")
    e = c / b
    if e != a:
        print(f"fail at {c} / {b} != {a}. == {d}")
        return False
    else:
        print("pass")
    return True

a, b = setup()
test(a, b)

In [ ]:
for _ in range(100):
    a,b = setup()
    if not test(a, b):
        break

In [ ]:
p + q + r

In [ ]:
p - q

In [ ]:
p * q

In [ ]:
# evaluate polynumber p at 2
# ie p(x) = 5 + 6x^3
p(2)

In [ ]:
# For p,q elements of Polynumber
# (pq)(c) == p(c)q(c)
# (p + q)(c) == p(c) + q(c)

(p - q)(2) == p(2) - q(2)

In [ ]:
2 * p